# Лабораторная 1. Среда, данные и первая сводка

Задание целиком: `labs/lab01/README.md` в репозитории курса.

Как работать с ноутбуком:
1. В первой ячейке впишите свой номер варианта.
2. Заполните ячейки с пометкой `TODO` по порядку. Подсказки в комментариях.
3. В конце выполните Run All с нуля, скачайте ноутбук и `results.zip`.

Ноутбук работает в Kaggle, Colab и локально. В Kaggle включите интернет: правая панель → Settings → Internet → On. Если интернет в Kaggle включить нельзя, работайте в Colab или локально.

## 0. Установка и загрузка данных

Ячейка определяет, где запущен ноутбук, находит или скачивает `get_data.py` из репозитория курса и загружает ваш вариант. Ничего в ней менять не нужно, кроме `VARIANT`.

In [ ]:
VARIANT = None   # TODO: ваш порядковый номер в списке группы, например 7
DATA_SOURCE = None  # запасной путь: папка с архивами variant_NN.zip

import os
import sys
import inspect
import urllib.request
from pathlib import Path

assert isinstance(VARIANT, int) and 0 <= VARIANT <= 22, "Впишите свой номер варианта: целое число от 1 до 22"
if VARIANT == 0:
    print("ВНИМАНИЕ: вариант 00 демонстрационный, работа на нём не засчитывается")

if "google.colab" in sys.modules:
    ENV, DATA_ROOT = "colab", Path("/content/data")
elif Path("/kaggle/working").exists():
    ENV, DATA_ROOT = "kaggle", Path("/kaggle/working/data")
elif Path("../../data/get_data.py").exists():
    # Ноутбук открыт прямо из клона репозитория курса: данные кладём в его папку data/
    ENV, DATA_ROOT = "local", Path("../../data").resolve()
else:
    # Локально: ноутбук лежит в cv-2026-familiya/lab01/, данные кладём в cv-2026-familiya/data/.
    # Эта папка указана в .gitignore, в репозиторий она не попадёт.
    ENV, DATA_ROOT = "local", Path("..").resolve() / "data"
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print("среда:", ENV, "| папка данных:", DATA_ROOT)

GET_DATA_URL = "https://raw.githubusercontent.com/ykvnkm/computer-vision-course/main/data/get_data.py"
course_copy = Path("../../data/get_data.py")
kaggle_copies = sorted(Path("/kaggle/input").glob("**/get_data.py")) if ENV == "kaggle" else []
if course_copy.exists():
    script_dir = course_copy.resolve().parent
elif kaggle_copies:
    # Kaggle с подключённым датасетом-зеркалом курса: интернет не нужен
    script_dir = kaggle_copies[0].parent
    print("get_data.py из подключённого датасета:", kaggle_copies[0])
else:
    script_dir = DATA_ROOT
    if not (script_dir / "get_data.py").exists():
        try:
            urllib.request.urlretrieve(GET_DATA_URL, script_dir / "get_data.py")
        except Exception as e:
            raise RuntimeError("Не удалось скачать get_data.py. В Kaggle включите Internet в Settings; "
                               "в остальных случаях проверьте подключение к сети.") from e
sys.path.insert(0, str(script_dir))
from get_data import ensure_variant

params = inspect.signature(ensure_variant).parameters
kwargs = {"dest": DATA_ROOT} if "dest" in params else {}
if DATA_SOURCE and "source" in params:
    kwargs["source"] = DATA_SOURCE
DATA_DIR = Path(ensure_variant(VARIANT, **kwargs))
print("данные варианта:", DATA_DIR)
print(sorted(p.name for p in DATA_DIR.iterdir()))

In [ ]:
import json
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image

IMG_DIR = DATA_DIR / "images"
XML_DIR = DATA_DIR / "annotations" / "xmls"
TRIMAP_DIR = DATA_DIR / "annotations" / "trimaps"
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)

print("OpenCV", cv2.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)

## 1. Что лежит в папке варианта

Прочитайте `variant.json` и `labels.csv`. Проверьте, что номер варианта в `variant.json` совпадает с вашим.

In [ ]:
# TODO: прочитайте variant.json (json.loads + read_text) и напечатайте его
variant_info = ...

# TODO: прочитайте labels.csv в DataFrame labels и покажите первые строки
labels = ...


## 2. Изображение как массив

Возьмите одно изображение и посмотрите, что вернул OpenCV. Выберите файл, который Pillow определяет как `JPEG` в режиме `RGB`: необычные файлы разберём в разделе 4.

In [ ]:
# Выбираем самое «цветное» фото среди 100 случайных обычных JPEG: на нём ошибка BGR/RGB заметнее
best, best_score = None, -1.0
for fn in labels["filename"].sample(frac=1, random_state=0).head(100):
    with Image.open(IMG_DIR / fn) as im:
        if im.format != "JPEG" or im.mode != "RGB":
            continue
        arr = np.asarray(im.resize((64, 64)), dtype=np.float32)[16:48, 16:48]  # центр кадра
    score = float(np.abs(arr[..., 0] - arr[..., 2]).mean())
    if score > best_score:
        best, best_score = fn, score
example_name = best
example_path = IMG_DIR / example_name
print(example_name)

# TODO: прочитайте файл через cv2.imread(str(example_path)) в переменную bgr
# TODO: напечатайте type, shape, dtype, min, max и значение пикселя в центре кадра
#       bgr[bgr.shape[0] // 2, bgr.shape[1] // 2]


In [ ]:
# TODO: переведите bgr в rgb через cv2.cvtColor(..., cv2.COLOR_BGR2RGB)
# TODO: покажите рядом bgr и rgb через plt.imshow. Что случилось с цветами?


Базовые преобразования. Подсказки:

| Что | Функция |
|---|---|
| оттенки серого | `cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)` |
| ресайз | `cv2.resize(rgb, (ширина, высота))` |
| кроп | `rgb[y0:y1, x0:x1]` |
| отражение | `cv2.flip(rgb, 1)` |
| float 0..1 | `rgb.astype(np.float32) / 255` |

In [ ]:
# TODO: получите gray, resized (224x224), crop (центральная половина кадра), flipped, as_float
# TODO: для каждого напечатайте shape, dtype, min, max


In [ ]:
# Ловушка uint8: запустите и объясните результат в выводах
a = np.array([[50, 200, 250]], dtype=np.uint8)   # «картинка» 1x3 пикселя
print("numpy a + 100:", a + 100)
print("cv2.add:      ", cv2.add(a, np.full_like(a, 100)))

In [ ]:
# TODO: соберите 8 панелей на одной фигуре 2x4:
#   BGR как есть, RGB, канал R, канал B, серое, resize, кроп, отражение
# и сохраните: fig.savefig(RESULTS / "transforms.png", dpi=110)


## 3. Сводка по датасету

Пройдите по всем файлам в `images/`. Для каждого соберите словарь и добавьте в список `rows`.

Почему два инструмента:
- `Image.open(p)` из Pillow честно говорит формат (`.format`) и режим (`.mode`: `RGB`, `L`, `RGBA`, `P`, ...);
- `cv2.imread(p)` по умолчанию всегда возвращает 3 канала, а для нечитаемого файла возвращает `None` без ошибки. Для проверки каналов читайте с флагом `cv2.IMREAD_UNCHANGED`.

In [ ]:
label_by_file = labels.set_index("filename")

rows = []
for p in sorted(IMG_DIR.iterdir()):
    if p.name.startswith("."):
        continue
    rec = {"filename": p.name, "ext": p.suffix.lower(), "file_kb": round(p.stat().st_size / 1024, 1)}

    # TODO: через Image.open(p) запишите pil_format, mode, width, height
    #       (оберните в try/except: не каждый файл в папке окажется картинкой)

    # TODO: raw = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
    #       запишите cv2_ok (raw is not None) и cv2_channels (1, если raw.ndim == 2, иначе raw.shape[2])

    # TODO: средняя яркость: cv2.imread(..., cv2.IMREAD_GRAYSCALE).mean()

    # TODO: has_xml и has_trimap: существуют ли XML_DIR / (p.stem + ".xml") и TRIMAP_DIR / (p.stem + ".png")

    # TODO: breed и species из label_by_file (если файла нет в labels.csv, поставьте None)

    rows.append(rec)

info = pd.DataFrame(rows)
# TODO: добавьте колонку aspect = width / height
info.to_csv(RESULTS / "images_info.csv", index=False)
info.head()

In [ ]:
# TODO: соберите словарь summary с главными числами и сохраните в results/summary.csv
# Минимум: число файлов, пород, кошек и собак, самая крупная и самая малочисленная порода,
# min/медиана/max ширины и высоты, число уникальных размеров, число альбомных и портретных,
# форматы и режимы (value_counts), сколько не RGB, сколько .jpg не JPEG, сколько не читаются,
# сколько с XML и trimap, средняя яркость.
summary = {
    "variant": VARIANT,
    # ...
}
summary_df = pd.DataFrame(list(summary.items()), columns=["metric", "value"])
summary_df.to_csv(RESULTS / "summary.csv", index=False)
summary_df

In [ ]:
# TODO: таблица по породам: groupby(["breed", "species"]) и agg:
#   n_images, n_with_xml, n_not_rgb, width_median, height_median, brightness_mean
# сохраните в results/per_breed.csv


In [ ]:
# TODO: три гистограммы на одной фигуре: ширина, высота, соотношение сторон
# сохраните в results/sizes_hist.png


## 4. Проблемные файлы

Найдите файлы, которые сломают обучение в следующих лабах:
- режим не `RGB`;
- расширение `.jpg`, а Pillow говорит другой формат;
- файл не читается ни Pillow, ни OpenCV;
- нет метки в `labels.csv` или нет trimap.

Для двух-трёх таких файлов сравните `cv2.imread(p).shape` и `cv2.imread(p, cv2.IMREAD_UNCHANGED).shape`.

In [ ]:
# TODO: отфильтруйте info и напечатайте каждый список проблемных файлов


In [ ]:
# TODO: сравните формы массивов для нескольких проблемных файлов


In [ ]:
def md_table(df):
    # DataFrame -> таблица Markdown
    if df.empty:
        return "нет\n"
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join(str(r[c]) for c in cols) + " |")
    return "\n".join(lines) + "\n"

# TODO: соберите текст отчёта (заголовок, режимы, списки проблемных файлов, разметка)
report = [
    f"# Отчёт о каналах и проблемных файлах, вариант {VARIANT:02d}",
    "",
    # ...
]
(RESULTS / "channels_report.md").write_text("\n".join(report), encoding="utf-8")

## 5. Разметка: бокс головы и trimap

- XML в формате Pascal VOC: внутри `<object><bndbox>` координаты `xmin, ymin, xmax, ymax`. Есть не у всех изображений. Обычно объект один, но бывает и два: берите все `<object>`.
- Trimap: PNG того же размера, что и фото. 1 = животное, 2 = фон, 3 = граница.

In [ ]:
def read_boxes(xml_path):
    # Все боксы из XML: список (xmin, ymin, xmax, ymax) и размер (w, h), записанный в XML
    root = ET.parse(xml_path).getroot()
    size = root.find("size")
    w, h = int(size.find("width").text), int(size.find("height").text)
    boxes = [tuple(int(float(bb.find(k).text)) for k in ("xmin", "ymin", "xmax", "ymax"))
             for bb in root.iter("bndbox")]
    return boxes, (w, h)

# TODO: возьмите одно изображение с XML, напечатайте его боксы
# TODO: прочитайте его trimap с cv2.IMREAD_UNCHANGED, выведите np.unique(trimap, return_counts=True)
# TODO: покажите фото и trimap рядом (для trimap: cmap="viridis", vmin=1, vmax=3)


## 6. Сетка примеров

По 2 изображения каждой породы. На каждом нарисуйте:
- бокс головы: `cv2.rectangle(img, (x0, y0), (x1, y1), (0, 255, 0), 3)`;
- контур животного: маска `(trimap == 1).astype(np.uint8)`, затем `cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)` и `cv2.drawContours(img, contours, -1, (255, 220, 0), 3)`.

Используйте `np.random.default_rng(0)`, чтобы при перезапуске сетка не менялась.

In [ ]:
# TODO: соберите сетку и сохраните в results/samples_grid.png


## 7. Числа для отчёта и выгрузка

Напечатайте всё, что понадобится в README. Затем упакуйте `results/` в архив и скачайте:
- в Kaggle: правая панель → Output → `results.zip` → Download; ноутбук: File → Download Notebook;
- в Colab: архив скачается сам после запуска ячейки; ноутбук: File → Download → Download .ipynb;
- локально: ничего скачивать не нужно, всё уже лежит в папке `lab01/`.

Переименуйте скачанный ноутбук в `solution.ipynb`.

In [ ]:
print(summary_df.to_string(index=False))
print("артефакты:", sorted(p.name for p in RESULTS.iterdir()))

expected = {"summary.csv", "per_breed.csv", "images_info.csv", "channels_report.md",
            "transforms.png", "sizes_hist.png", "samples_grid.png"}
missing = expected - {p.name for p in RESULTS.iterdir()}
print("не хватает:", missing if missing else "ничего, все 7 файлов на месте")

In [ ]:
import shutil
if ENV != "local":
    archive = shutil.make_archive("results", "zip", RESULTS)
    print("архив:", archive)
    if ENV == "colab":
        from google.colab import files
        files.download(archive)